In [35]:
import duckdb

con = duckdb.connect("../data/silver/silver.duckdb")

con.sql("SHOW TABLES").df()

,name
0,silver_attack_regions
1,silver_attacks
2,silver_weapon_reference


In [36]:
con.sql("DESCRIBE silver_attacks").df()


,column_name,column_type,null,key,default,extra
0,source_file,VARCHAR,YES,None,None,None
1,source_row_number,BIGINT,YES,None,None,None
2,event_date,VARCHAR,YES,None,None,None
3,time_start,VARCHAR,YES,None,None,None
4,time_end,VARCHAR,YES,None,None,None
5,weapon_model,VARCHAR,YES,None,None,None
6,weapon_model_key,VARCHAR,YES,None,None,None
7,weapon_category,VARCHAR,YES,None,None,None
8,weapon_type,VARCHAR,YES,None,None,None
9,weapon_national_origin,VARCHAR,YES,None,None,None


In [38]:
con.sql("""
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'main'
ORDER BY table_name, ordinal_position
""").df()


,table_name,column_name,data_type
0,silver_attack_regions,source_file,VARCHAR
1,silver_attack_regions,source_row_number,BIGINT
2,silver_attack_regions,event_date,VARCHAR
3,silver_attack_regions,time_start,VARCHAR
4,silver_attack_regions,time_end,VARCHAR
...,...,...,...
78,silver_weapon_reference,in_service,VARCHAR
79,silver_weapon_reference,designer,VARCHAR
80,silver_weapon_reference,manufacturer,VARCHAR
81,silver_weapon_reference,guidance_system,VARCHAR


In [39]:
con.sql("""
SELECT 'silver_attacks' AS table_name, COUNT(*) AS rows FROM silver_attacks
UNION ALL
SELECT 'silver_attack_regions', COUNT(*) FROM silver_attack_regions
UNION ALL
SELECT 'silver_weapon_reference', COUNT(*) FROM silver_weapon_reference
""").df()


,table_name,rows
0,silver_attacks,3597
1,silver_attack_regions,3715
2,silver_weapon_reference,64


In [40]:
import duckdb
import pandas as pd
from IPython.display import display

DB_PATH = "../data/silver/silver.duckdb"
con = duckdb.connect(DB_PATH)

tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
      AND table_type = 'BASE TABLE'
    ORDER BY table_name
""").df()["table_name"].tolist()

summary_rows = []

for table_name in tables:
    row_count = con.execute(f'SELECT COUNT(*) AS n FROM "{table_name}"').fetchone()[0]

    cols = con.execute(f"""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'main'
          AND table_name = '{table_name}'
        ORDER BY ordinal_position
    """).df()

    summary_rows.append({
        "table_name": table_name,
        "row_count": row_count,
        "column_count": len(cols),
        "columns": ", ".join(
            f"{col} ({dtype})"
            for col, dtype in zip(cols["column_name"], cols["data_type"])
        )
    })

schema_summary = pd.DataFrame(summary_rows)
display(schema_summary)


,table_name,row_count,column_count,columns
0,silver_attack_regions,3715,35,"source_file (VARCHAR), source_row_number (BIGI..."
1,silver_attacks,3597,33,"source_file (VARCHAR), source_row_number (BIGI..."
2,silver_weapon_reference,64,15,"source_file (VARCHAR), source_row_number (BIGI..."


## Exploration visuals and checks

These cells focus on data quality, area coverage, weapon models, and first trend cuts from the silver DuckDB tables.


In [41]:
import duckdb
import pandas as pd
import plotly.express as px
from IPython.display import display

DB_PATH = "../data/silver/silver.duckdb"
con = duckdb.connect(DB_PATH)


### Quality summary


In [42]:
quality_summary = con.sql("""
SELECT
    COUNT(*) AS total_attack_rows,
    MIN(CAST(NULLIF(event_date, '') AS DATE)) AS min_event_date,
    MAX(CAST(NULLIF(event_date, '') AS DATE)) AS max_event_date,
    COUNT(DISTINCT weapon_model) AS distinct_weapon_models,
    SUM(CASE WHEN COALESCE(target, '') = '' THEN 1 ELSE 0 END) AS empty_target_rows,
    SUM(CASE WHEN COALESCE(NULLIF(area_macro, ''), 'unknown') = 'unknown' THEN 1 ELSE 0 END) AS unknown_area_macro_rows,
    SUM(CASE WHEN NOT weapon_reference_match THEN 1 ELSE 0 END) AS unmatched_weapon_reference_rows
FROM silver_attacks
""").df()

display(quality_summary)


,total_attack_rows,min_event_date,max_event_date,distinct_weapon_models,empty_target_rows,unknown_area_macro_rows,unmatched_weapon_reference_rows
0,3597,2022-09-28,2026-04-10,71,30.0,30.0,9.0


### Area macro distribution


In [43]:
unknown_target_summary = con.sql("""
SELECT
    SUM(CASE WHEN COALESCE(target, '') = '' THEN 1 ELSE 0 END) AS empty_target_rows,
    SUM(CASE WHEN COALESCE(NULLIF(area_macro, ''), 'unknown') = 'unknown' THEN 1 ELSE 0 END) AS unknown_area_macro_rows
FROM silver_attacks
""").df()

display(unknown_target_summary)

area_macro_summary = con.sql("""
SELECT
    COALESCE(NULLIF(area_macro, ''), 'unknown') AS area_macro,
    COUNT(*) AS attack_rows,
    SUM(COALESCE(launched, 0)) AS launched_total,
    SUM(COALESCE(destroyed, 0)) AS destroyed_total
FROM silver_attacks
GROUP BY 1
ORDER BY launched_total DESC, attack_rows DESC
""").df()

display(area_macro_summary)


,empty_target_rows,unknown_area_macro_rows
0,30.0,30.0


,area_macro,attack_rows,launched_total,destroyed_total
0,nationwide,1485,90589.0,65050.0
1,south,1499,7057.0,6549.0
2,multi,66,1116.0,776.0
3,north,60,600.0,535.0
4,east,188,576.0,272.0
5,center-east,175,314.0,214.0
6,unknown,30,192.0,133.0
7,west,16,126.0,107.0
8,south-east,34,100.0,62.0
9,center-west,14,73.0,51.0


In [44]:
fig = px.bar(
    area_macro_summary,
    x="area_macro",
    y="launched_total",
    text="attack_rows",
    title="Launched total by area macro (text = attack rows)"
)
fig.update_layout(xaxis_title="Area macro", yaxis_title="Launched total")
fig.show()


In [45]:
area_macro_log = area_macro_summary.query("launched_total > 0").copy()

fig = px.bar(
    area_macro_log,
    x="area_macro",
    y="launched_total",
    text="attack_rows",
    title="Launched total by area macro (log scale, text = attack rows)"
)
fig.update_layout(xaxis_title="Area macro", yaxis_title="Launched total (log scale)")
fig.update_yaxes(type="log")
fig.show()


### Weapon model distribution


In [46]:
weapon_model_summary = con.sql("""
WITH model_summary AS (
    SELECT
        CASE
            WHEN COALESCE(NULLIF(TRIM(weapon_model), ''), '') = '' THEN 'Unknown'
            WHEN LOWER(weapon_model) LIKE 'unknown%' THEN 'Unknown'
            ELSE weapon_model
        END AS weapon_model_group,
        COALESCE(NULLIF(weapon_category, ''), 'Unknown category') AS weapon_category,
        SUM(COALESCE(launched, 0)) AS launched_total,
        COUNT(*) AS attack_rows
    FROM silver_attacks
    GROUP BY 1, 2
),
ranked_non_unknown AS (
    SELECT *
    FROM model_summary
    WHERE weapon_model_group <> 'Unknown'
    ORDER BY launched_total DESC, attack_rows DESC, weapon_model_group
    LIMIT 19
),
unknown_row AS (
    SELECT *
    FROM model_summary
    WHERE weapon_model_group = 'Unknown'
)
SELECT * FROM ranked_non_unknown
UNION ALL
SELECT * FROM unknown_row
ORDER BY launched_total DESC, attack_rows DESC, weapon_model_group
""").df()

display(weapon_model_summary)

fig = px.bar(
    weapon_model_summary.sort_values('launched_total', ascending=True),
    x="launched_total",
    y="weapon_model_group",
    color="weapon_category",
    orientation="h",
    title="Top weapon models by launched total (including Unknown)"
)
fig.update_layout(xaxis_title="Launched total", yaxis_title="Weapon model")
fig.show()


,weapon_model_group,weapon_category,launched_total,attack_rows
0,Shahed-136/131,UAV,87875.0,991
1,Unknown,UAV,3381.0,264
2,X-101/X-555,cruise missile,1994.0,102
3,Молнія,UAV,811.0,112
4,X-101/X-555 and Kalibr,cruise missile,643.0,9
5,Kalibr,cruise missile,557.0,73
6,Iskander-M,ballistic missile,482.0,264
7,Reconnaissance UAV,UAV,460.0,200
8,Iskander-M/KN-23,ballistic missile,364.0,83
9,Lancet,UAV,355.0,143


In [47]:
coverage_summary = con.sql("""
SELECT
    MIN(CAST(NULLIF(event_date, '') AS DATE)) AS min_event_date,
    MAX(CAST(NULLIF(event_date, '') AS DATE)) AS max_event_date,
    COUNT(*) AS total_attack_rows,
    COUNT(DISTINCT event_date) AS distinct_event_dates,
    COUNT(DISTINCT weapon_model) AS distinct_weapon_models,
    COUNT(DISTINCT area_macro) AS distinct_area_macros
FROM silver_attacks
""").df()

display(coverage_summary)


,min_event_date,max_event_date,total_attack_rows,distinct_event_dates,distinct_weapon_models,distinct_area_macros
0,2022-09-28,2026-04-10,3597,1066,71,14


In [48]:
daily_trend = con.sql("""
WITH daily AS (
    SELECT
        CAST(event_date AS DATE) AS event_date,
        SUM(COALESCE(launched, 0)) AS launched_total,
        SUM(COALESCE(destroyed, 0)) AS destroyed_total
    FROM silver_attacks
    WHERE event_date <> ''
    GROUP BY 1
)
SELECT
    event_date,
    launched_total,
    destroyed_total,
    AVG(launched_total) OVER (
        ORDER BY event_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS launched_7d_avg
FROM daily
ORDER BY event_date
""").df()

daily_long = daily_trend.melt(
    id_vars='event_date',
    value_vars=['launched_total', 'destroyed_total', 'launched_7d_avg'],
    var_name='metric',
    value_name='value'
)

fig = px.line(
    daily_long,
    x='event_date',
    y='value',
    color='metric',
    title='Daily launched vs destroyed with 7-day launched average'
)
fig.update_layout(xaxis_title='Event date', yaxis_title='Count')
fig.show()


In [49]:
monthly_trend = con.sql("""
SELECT
    DATE_TRUNC('month', CAST(event_date AS DATE)) AS month_start,
    SUM(COALESCE(launched, 0)) AS launched_total,
    SUM(COALESCE(destroyed, 0)) AS destroyed_total,
    COUNT(*) AS attack_rows
FROM silver_attacks
WHERE event_date <> ''
GROUP BY 1
ORDER BY 1
""").df()

display(monthly_trend.tail(12))

monthly_long = monthly_trend.melt(
    id_vars='month_start',
    value_vars=['launched_total', 'destroyed_total'],
    var_name='metric',
    value_name='value'
)

fig = px.bar(
    monthly_long,
    x='month_start',
    y='value',
    color='metric',
    barmode='group',
    title='Monthly launched vs destroyed totals'
)
fig.update_layout(xaxis_title='Month', yaxis_title='Count')
fig.show()


,month_start,launched_total,destroyed_total,attack_rows
32,2025-05-01,4574.0,2196.0,102
33,2025-06-01,5374.0,2779.0,121
34,2025-07-01,6688.0,3852.0,123
35,2025-08-01,4462.0,3729.0,105
36,2025-09-01,6022.0,5220.0,101
37,2025-10-01,5932.0,4704.0,107
38,2025-11-01,5802.0,4791.0,104
39,2025-12-01,5830.0,4772.0,90
40,2026-01-01,4979.0,4188.0,101
41,2026-02-01,6030.0,5253.0,120


In [50]:
region_summary = con.sql("""
SELECT
    area_region,
    COUNT(*) AS attack_rows,
    SUM(COALESCE(launched, 0)) AS launched_total
FROM silver_attack_regions
WHERE has_specific_area_region
GROUP BY 1
ORDER BY attack_rows DESC, launched_total DESC
LIMIT 20
""").df()

display(region_summary)

fig = px.bar(
    region_summary.sort_values('attack_rows', ascending=True),
    x='attack_rows',
    y='area_region',
    orientation='h',
    title='Top specific area regions by attack rows'
)
fig.update_layout(xaxis_title='Attack rows', yaxis_title='Area region')
fig.show()


,area_region,attack_rows,launched_total
0,Odesa oblast,244,1353.0
1,Kherson oblast,171,345.0
2,Mykolaiv oblast,159,695.0
3,Dnipropetrovsk oblast,148,470.0
4,Kharkiv oblast,81,321.0
5,Kyiv oblast,51,816.0
6,Zaporizhzhia oblast,38,223.0
7,Sumy oblast,36,244.0
8,Poltava oblast,30,135.0
9,Donetsk oblast,26,86.0


In [51]:
unmatched_models = con.sql("""
SELECT
    weapon_model,
    COUNT(*) AS attack_rows,
    SUM(COALESCE(launched, 0)) AS launched_total
FROM silver_attacks
WHERE NOT weapon_reference_match
GROUP BY 1
ORDER BY launched_total DESC, attack_rows DESC
""").df()

display(unmatched_models)


,weapon_model,attack_rows,launched_total
0,Iskander-M/KN-23 and Iskander-K and Kalibr,1,38.0
1,Iskander-M/KN-23 and Iskander-K and X-59/X-69,1,27.0
2,Aerial Bomb,1,16.0
3,X-101/X-555 and Kalibr and Iskander-M/KN-23,1,14.0
4,Iskander-M and Iskander-K,2,13.0
5,Привет-82,1,1.0
6,Фенікс,1,1.0
7,Картограф,1,1.0


In [52]:
quarterly_top_models = con.sql("""
WITH quarterly AS (
    SELECT
        DATE_TRUNC('quarter', CAST(event_date AS DATE)) AS quarter_start,
        COALESCE(NULLIF(weapon_model, ''), 'Unknown model') AS weapon_model,
        COALESCE(NULLIF(weapon_category, ''), 'Unknown category') AS weapon_category,
        SUM(COALESCE(launched, 0)) AS launched_total,
        COUNT(*) AS attack_rows
    FROM silver_attacks
    WHERE event_date <> ''
    GROUP BY 1, 2, 3
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY quarter_start
            ORDER BY launched_total DESC, attack_rows DESC, weapon_model
        ) AS rank_in_quarter
    FROM quarterly
)
SELECT *
FROM ranked
WHERE rank_in_quarter <= 5
ORDER BY quarter_start DESC, rank_in_quarter
""").df()

display(quarterly_top_models)

fig = px.bar(
    quarterly_top_models.sort_values(["quarter_start", "rank_in_quarter"]),
    x="quarter_start",
    y="launched_total",
    color="weapon_model",
    barmode="group",
    hover_data=["weapon_category", "attack_rows", "rank_in_quarter"],
    title="Top 5 weapon models by quarter (launched total)"
)
fig.update_layout(xaxis_title="Quarter", yaxis_title="Launched total")
fig.show()


,quarter_start,weapon_model,weapon_category,launched_total,attack_rows,rank_in_quarter
0,2026-04-01,Shahed-136/131,UAV,2288.0,11,1
1,2026-04-01,Unknown UAV,UAV,593.0,9,2
2,2026-04-01,X-101/X-555,cruise missile,25.0,1,3
3,2026-04-01,Iskander-M,ballistic missile,10.0,1,4
4,2026-04-01,Iskander-K,cruise missile,2.0,1,5
...,...,...,...,...,...,...
74,2022-10-01,Kalibr,cruise missile,51.0,6,5
75,2022-07-01,Shahed-136/131,UAV,7.0,1,1
76,2022-07-01,X-59,cruise missile,5.0,1,2
77,2022-07-01,Orlan-10,UAV,3.0,1,3


In [53]:
monthly_model_heatmap = con.sql("""
WITH top_models AS (
    SELECT
        COALESCE(NULLIF(weapon_model, ''), 'Unknown model') AS weapon_model
    FROM silver_attacks
    GROUP BY 1
    ORDER BY SUM(COALESCE(launched, 0)) DESC
    LIMIT 12
),
monthly AS (
    SELECT
        DATE_TRUNC('month', CAST(event_date AS DATE)) AS month_start,
        COALESCE(NULLIF(weapon_model, ''), 'Unknown model') AS weapon_model,
        SUM(COALESCE(launched, 0)) AS launched_total
    FROM silver_attacks
    WHERE event_date <> ''
      AND COALESCE(NULLIF(weapon_model, ''), 'Unknown model') IN (SELECT weapon_model FROM top_models)
    GROUP BY 1, 2
)
SELECT *
FROM monthly
ORDER BY month_start, launched_total DESC
""").df()

fig = px.density_heatmap(
    monthly_model_heatmap,
    x="month_start",
    y="weapon_model",
    z="launched_total",
    histfunc="sum",
    color_continuous_scale="Reds",
    title="Monthly launched total heatmap for top weapon models"
)
fig.update_layout(xaxis_title="Month", yaxis_title="Weapon model")
fig.show()


In [54]:
launch_scope_summary = con.sql("""
SELECT
    CASE
        WHEN area_macro = 'nationwide' THEN 'nationwide'
        WHEN area_count > 0 THEN 'specific_area_rows'
        WHEN COALESCE(NULLIF(area_macro, ''), 'unknown') = 'unknown' THEN 'unknown'
        ELSE 'directional_or_other_macro'
    END AS target_scope,
    COUNT(*) AS attack_rows,
    SUM(COALESCE(launched, 0)) AS launched_total
FROM silver_attacks
GROUP BY 1
ORDER BY launched_total DESC, attack_rows DESC
""").df()

launch_scope_summary["launched_share_pct"] = (
    100 * launch_scope_summary["launched_total"] / launch_scope_summary["launched_total"].sum()
).round(2)

display(launch_scope_summary)

fig = px.bar(
    launch_scope_summary,
    x="target_scope",
    y="launched_total",
    text="launched_share_pct",
    title="Launched volume by target scope (text = share of launched total %)"
)
fig.update_layout(xaxis_title="Target scope", yaxis_title="Launched total")
fig.show()

fig = px.pie(
    launch_scope_summary,
    names="target_scope",
    values="launched_total",
    title="Share of launched total by target scope"
)
fig.show()


,target_scope,attack_rows,launched_total,launched_share_pct
0,nationwide,1485,90589.0,89.85
1,directional_or_other_macro,1098,6320.0,6.27
2,specific_area_rows,984,3724.0,3.69
3,unknown,30,192.0,0.19


In [55]:
directional_macro_summary = con.sql("""
SELECT
    area_macro,
    COUNT(*) AS attack_rows,
    SUM(COALESCE(launched, 0)) AS launched_total,
    SUM(COALESCE(destroyed, 0)) AS destroyed_total
FROM silver_attacks
WHERE area_macro IN (
    'north', 'south', 'east', 'west', 'center',
    'north-east', 'south-east', 'center-east', 'center-west'
)
GROUP BY 1
ORDER BY launched_total DESC, attack_rows DESC
""").df()

display(directional_macro_summary)

fig = px.bar(
    directional_macro_summary,
    x='area_macro',
    y='launched_total',
    text='attack_rows',
    color='launched_total',
    color_continuous_scale=['#2b0a0a', '#5e1111', '#962020', '#d73a31'],
    title='Directional macro areas by launched total (text = attack rows)'
)
fig.update_layout(xaxis_title='Area macro direction', yaxis_title='Launched total')
fig.show()


,area_macro,attack_rows,launched_total,destroyed_total
0,south,1499,7057.0,6549.0
1,north,60,600.0,535.0
2,east,188,576.0,272.0
3,center-east,175,314.0,214.0
4,west,16,126.0,107.0
5,south-east,34,100.0,62.0
6,center-west,14,73.0,51.0
7,north-east,22,59.0,30.0
8,center,4,12.0,10.0


In [56]:
direction_centroids = pd.DataFrame([
    {'area_macro': 'north', 'lat': 51.0, 'lon': 31.5},
    {'area_macro': 'north-east', 'lat': 50.8, 'lon': 35.4},
    {'area_macro': 'east', 'lat': 49.0, 'lon': 36.8},
    {'area_macro': 'south-east', 'lat': 47.7, 'lon': 35.5},
    {'area_macro': 'south', 'lat': 46.8, 'lon': 31.5},
    {'area_macro': 'center', 'lat': 49.0, 'lon': 31.5},
    {'area_macro': 'center-east', 'lat': 48.9, 'lon': 34.2},
    {'area_macro': 'center-west', 'lat': 49.2, 'lon': 27.5},
    {'area_macro': 'west', 'lat': 49.5, 'lon': 24.5},
])

directional_map = directional_macro_summary.merge(direction_centroids, on='area_macro', how='left')

fig = px.scatter_geo(
    directional_map,
    lat='lat',
    lon='lon',
    text='area_macro',
    size='attack_rows',
    size_max=48,
    color='launched_total',
    hover_name='area_macro',
    hover_data={
        'attack_rows': True,
        'launched_total': True,
        'destroyed_total': True,
        'lat': False,
        'lon': False,
    },
    color_continuous_scale=['#220707', '#5a1010', '#9e1e1e', '#ff7043'],
    projection='mercator',
    scope='europe',
    title='Directional macro target map of Ukraine'
)
fig.update_traces(
    marker=dict(line=dict(color='black', width=1.0), opacity=0.9, sizemin=14),
    textposition='top center'
)
fig.update_geos(
    center={'lat': 48.8, 'lon': 31.5},
    lataxis_range=[44, 53],
    lonaxis_range=[22, 41],
    showcountries=True,
    countrycolor='rgb(90, 90, 90)',
    showland=True,
    landcolor='rgb(239, 236, 230)',
    showocean=True,
    oceancolor='rgb(222, 228, 236)'
)
fig.show()


### 3. Which models have the weakest reference coverage?


In [57]:
reference_coverage_summary = con.sql("""
SELECT
    COUNT(*) AS total_attack_rows,
    SUM(CASE WHEN weapon_reference_match THEN 1 ELSE 0 END) AS matched_rows,
    SUM(CASE WHEN NOT weapon_reference_match THEN 1 ELSE 0 END) AS unmatched_rows,
    SUM(CASE WHEN weapon_reference_match THEN COALESCE(launched, 0) ELSE 0 END) AS matched_launched_total,
    SUM(CASE WHEN NOT weapon_reference_match THEN COALESCE(launched, 0) ELSE 0 END) AS unmatched_launched_total
FROM silver_attacks
""").df()

display(reference_coverage_summary)

unmatched_model_summary = con.sql("""
SELECT
    COALESCE(NULLIF(weapon_model, ''), 'Unknown model') AS weapon_model,
    COUNT(*) AS attack_rows,
    SUM(COALESCE(launched, 0)) AS launched_total,
    MIN(CAST(NULLIF(event_date, '') AS DATE)) AS first_seen,
    MAX(CAST(NULLIF(event_date, '') AS DATE)) AS last_seen
FROM silver_attacks
WHERE NOT weapon_reference_match
GROUP BY 1
ORDER BY launched_total DESC, attack_rows DESC, weapon_model
""").df()

display(unmatched_model_summary)

if not unmatched_model_summary.empty:
    fig = px.bar(
        unmatched_model_summary.sort_values("launched_total", ascending=True),
        x="launched_total",
        y="weapon_model",
        orientation="h",
        title="Unmatched weapon models by launched total"
    )
    fig.update_layout(xaxis_title="Launched total", yaxis_title="Weapon model")
    fig.show()


,total_attack_rows,matched_rows,unmatched_rows,matched_launched_total,unmatched_launched_total
0,3597,3588.0,9.0,100714.0,111.0


,weapon_model,attack_rows,launched_total,first_seen,last_seen
0,Iskander-M/KN-23 and Iskander-K and Kalibr,1,38.0,2025-11-07,2025-11-07
1,Iskander-M/KN-23 and Iskander-K and X-59/X-69,1,27.0,2025-07-25,2025-07-25
2,Aerial Bomb,1,16.0,2024-10-15,2024-10-15
3,X-101/X-555 and Kalibr and Iskander-M/KN-23,1,14.0,2025-02-20,2025-02-20
4,Iskander-M and Iskander-K,2,13.0,2023-05-29,2024-08-25
5,Картограф,1,1.0,2024-07-28,2024-07-28
6,Привет-82,1,1.0,2024-07-21,2024-07-21
7,Фенікс,1,1.0,2024-11-25,2024-11-25


In [58]:
region_activity = con.sql("""
SELECT
    area_region,
    COUNT(*) AS attack_rows,
    COUNT(DISTINCT event_date) AS active_days,
    SUM(COALESCE(launched, 0)) AS launched_total_exploded
FROM silver_attack_regions
WHERE has_specific_area_region
  AND area_region <> ''
GROUP BY 1
ORDER BY attack_rows DESC, launched_total_exploded DESC
""").df()

display(region_activity.head(20))

top_region_frequency = region_activity.nlargest(15, "attack_rows").sort_values("attack_rows", ascending=True)
fig = px.bar(
    top_region_frequency,
    x="attack_rows",
    y="area_region",
    orientation="h",
    title="Top specific regions by attack rows"
)
fig.update_layout(xaxis_title="Attack rows", yaxis_title="Area region")
fig.show()

top_region_launched = region_activity.nlargest(15, "launched_total_exploded").sort_values("launched_total_exploded", ascending=True)
fig = px.bar(
    top_region_launched,
    x="launched_total_exploded",
    y="area_region",
    orientation="h",
    title="Top specific regions by exploded launched total"
)
fig.update_layout(xaxis_title="Exploded launched total", yaxis_title="Area region")
fig.show()


,area_region,attack_rows,active_days,launched_total_exploded
0,Odesa oblast,244,201,1353.0
1,Kherson oblast,171,113,345.0
2,Mykolaiv oblast,159,132,695.0
3,Dnipropetrovsk oblast,148,125,470.0
4,Kharkiv oblast,81,63,321.0
5,Kyiv oblast,51,37,816.0
6,Zaporizhzhia oblast,38,31,223.0
7,Sumy oblast,36,29,244.0
8,Poltava oblast,30,25,135.0
9,Donetsk oblast,26,26,86.0


In [59]:
area_centroids = pd.DataFrame([
    {"area_region": "Odesa oblast", "lat": 46.48, "lon": 30.72, "area_kind": "oblast"},
    {"area_region": "Mykolaiv oblast", "lat": 46.97, "lon": 31.99, "area_kind": "oblast"},
    {"area_region": "Kyiv oblast", "lat": 50.45, "lon": 30.52, "area_kind": "oblast"},
    {"area_region": "Lviv oblast", "lat": 49.84, "lon": 24.03, "area_kind": "oblast"},
    {"area_region": "Rivne oblast", "lat": 50.62, "lon": 26.25, "area_kind": "oblast"},
    {"area_region": "Volyn oblast", "lat": 50.75, "lon": 25.33, "area_kind": "oblast"},
    {"area_region": "Kherson oblast", "lat": 46.64, "lon": 32.62, "area_kind": "oblast"},
    {"area_region": "Khmelnytskyi oblast", "lat": 49.42, "lon": 26.99, "area_kind": "oblast"},
    {"area_region": "Poltava oblast", "lat": 49.59, "lon": 34.55, "area_kind": "oblast"},
    {"area_region": "Kharkiv oblast", "lat": 49.99, "lon": 36.23, "area_kind": "oblast"},
    {"area_region": "Sumy oblast", "lat": 50.91, "lon": 34.80, "area_kind": "oblast"},
    {"area_region": "Chernihiv oblast", "lat": 51.50, "lon": 31.29, "area_kind": "oblast"},
    {"area_region": "Zaporizhzhia oblast", "lat": 47.84, "lon": 35.14, "area_kind": "oblast"},
    {"area_region": "Dnipropetrovsk oblast", "lat": 48.47, "lon": 35.04, "area_kind": "oblast"},
    {"area_region": "Donetsk oblast", "lat": 48.72, "lon": 37.55, "area_kind": "oblast"},
    {"area_region": "Kirovohrad oblast", "lat": 48.51, "lon": 32.26, "area_kind": "oblast"},
    {"area_region": "Vinnytsia oblast", "lat": 49.23, "lon": 28.48, "area_kind": "oblast"},
    {"area_region": "Cherkasy oblast", "lat": 49.44, "lon": 32.06, "area_kind": "oblast"},
    {"area_region": "Kyiv", "lat": 50.45, "lon": 30.52, "area_kind": "city"},
    {"area_region": "Odesa", "lat": 46.48, "lon": 30.72, "area_kind": "city"},
    {"area_region": "Kharkiv", "lat": 49.99, "lon": 36.23, "area_kind": "city"},
    {"area_region": "Kherson", "lat": 46.64, "lon": 32.62, "area_kind": "city"},
    {"area_region": "Dnipro", "lat": 48.47, "lon": 35.04, "area_kind": "city"},
    {"area_region": "Zaporizhzhia", "lat": 47.84, "lon": 35.14, "area_kind": "city"},
    {"area_region": "Sumy", "lat": 50.91, "lon": 34.80, "area_kind": "city"},
    {"area_region": "Kramatorsk", "lat": 48.72, "lon": 37.55, "area_kind": "city"},
    {"area_region": "Kryvyi Rih", "lat": 47.91, "lon": 33.39, "area_kind": "city"},
    {"area_region": "Starokostiantyniv", "lat": 49.76, "lon": 27.22, "area_kind": "city"},
    {"area_region": "Kolomyia", "lat": 48.53, "lon": 25.04, "area_kind": "city"},
    {"area_region": "Ochakiv", "lat": 46.61, "lon": 31.54, "area_kind": "city"},
    {"area_region": "Snake Island", "lat": 45.26, "lon": 30.20, "area_kind": "special"}
])

region_map = region_activity.merge(area_centroids, on="area_region", how="left")
mapped_region_map = region_map[region_map["lat"].notna()].copy()
missing_region_coords = region_map[region_map["lat"].isna()].copy()

mapped_region_map["label"] = ""
if not mapped_region_map.empty:
    label_threshold = mapped_region_map["attack_rows"].quantile(0.85)
    mapped_region_map.loc[mapped_region_map["attack_rows"] >= label_threshold, "label"] = mapped_region_map["area_region"]

display(mapped_region_map.sort_values("attack_rows", ascending=False).head(20))
display(missing_region_coords[["area_region", "attack_rows", "launched_total_exploded"]])

fig = px.scatter_geo(
    mapped_region_map,
    lat="lat",
    lon="lon",
    text="label",
    size="attack_rows",
    size_max=50,
    color="launched_total_exploded",
    hover_name="area_region",
    hover_data={
        "attack_rows": True,
        "active_days": True,
        "launched_total_exploded": True,
        "area_kind": True,
        "lat": False,
        "lon": False,
        "label": False,
    },
    color_continuous_scale=['#220707', '#5a1010', '#9e1e1e', '#ff7043'],
    projection="mercator",
    scope="europe",
    title="Ukraine region activity map (centroid bubbles, larger markers)"
)
fig.update_traces(
    marker=dict(line=dict(color='black', width=0.9), opacity=0.9, sizemin=12),
    textposition='top center'
)
fig.update_geos(
    center={"lat": 48.8, "lon": 31.5},
    lataxis_range=[44, 53],
    lonaxis_range=[22, 41],
    showcountries=True,
    countrycolor="rgb(90, 90, 90)",
    showland=True,
    landcolor="rgb(239, 236, 230)",
    showocean=True,
    oceancolor="rgb(222, 228, 236)"
)
fig.show()


,area_region,attack_rows,active_days,launched_total_exploded,lat,lon,area_kind,label
0,Odesa oblast,244,201,1353.0,46.48,30.72,oblast,Odesa oblast
1,Kherson oblast,171,113,345.0,46.64,32.62,oblast,Kherson oblast
2,Mykolaiv oblast,159,132,695.0,46.97,31.99,oblast,Mykolaiv oblast
3,Dnipropetrovsk oblast,148,125,470.0,48.47,35.04,oblast,Dnipropetrovsk oblast
4,Kharkiv oblast,81,63,321.0,49.99,36.23,oblast,Kharkiv oblast
5,Kyiv oblast,51,37,816.0,50.45,30.52,oblast,
6,Zaporizhzhia oblast,38,31,223.0,47.84,35.14,oblast,
7,Sumy oblast,36,29,244.0,50.91,34.80,oblast,
8,Poltava oblast,30,25,135.0,49.59,34.55,oblast,
9,Donetsk oblast,26,26,86.0,48.72,37.55,oblast,


,area_region,attack_rows,launched_total_exploded
23,Kursk oblast,3,12.0
